In [7]:
# !pip install py-trees

### concepts

- Composite Nodes (组合节点 - 控制流)：
    - Sequence (序列, ->)：AND 逻辑。依次执行子节点。
        - 如果某个子节点返回 FAILURE，则 Sequence 立即返回 FAILURE。
        - 只有所有子节点都返回 SUCCESS，Sequence 才返回 SUCCESS。
        - 如果子节点返回 RUNNING，Sequence 也会保持 RUNNING（卡在这里等该子节点完成）。
    - Selector (选择, ?)：OR 逻辑 / 备选逻辑。依次执行子节点。
        - 只要有一个子节点返回 SUCCESS，Selector 就立即返回 SUCCESS（不再执行后续）。
        - 只有所有子节点都返回 FAILURE，Selector 才返回 FAILURE。
- Leaf Nodes (叶子节点 - 执行逻辑)：
    - Action (动作)：执行具体任务（如“走路”、“抓取”）。返回状态：
        - RUNNING: 正在做（比如正在走路，还没到）。
        - SUCCESS: 做完了。
        - FAILURE: 做不到了。
    - Condition (条件)：检查状态（如“是否看见目标？”）。通常只返回 SUCCESS 或 FAILURE。
- Blackboard (黑板)：
    - 所有节点共享的内存区域，用于在不同节点间传递数据（例如：Node A 看见了一个物体，把坐标写在黑板上，Node B 读取坐标去抓取）。

### examples

任务目标：机器人需要去厨房取咖啡并送给主人。 约束条件：
- 优先级最高：如果电量低于 20%，必须立即停止当前任务去充电。
- 正常任务：导航到厨房 -> 抓取咖啡 -> 导航到沙发。
  
行为树的设计结构：这里使用 Selector（选择器/Fallback） 来处理优先级，使用 Sequence（序列） 来处理步骤。
- 根节点 (Selector - 优先级仲裁)
    - 分支 1 (Sequence - 电池保护):
        - [条件] 检查电池是否过低？
        - [动作] 去充电坞充电。
    - 分支 2 (Sequence - 送咖啡任务):
        - [动作] 导航到厨房。
        - [动作] 抓取咖啡杯。
        - [动作] 导航到沙发（用户位置）。

```mermaid
graph TD
    Root[("? <br/>(Selector)<br/>Robot Brain")]
    
    %% 定义样式
    style Root fill:#f9f,stroke:#333,stroke-width:2px
    
    %% 第一层分支
    SeqBat[("-> <br/>(Sequence)<br/>Battery Guard")]
    SeqWork[("-> <br/>(Sequence)<br/>Deliver Coffee")]
    
    %% 连接 Root
    Root --> SeqBat
    Root --> SeqWork
    
    %% 电池分支的子节点
    CondBat([Condition:<br/>Check Battery < 20%?])
    ActCharge[Action:<br/>Charge Robot]
    
    SeqBat --> CondBat
    SeqBat --> ActCharge
    
    %% 工作分支的子节点
    ActMove1[Action:<br/>Go to Kitchen]
    ActGrab[Action:<br/>Pick Coffee]
    ActMove2[Action:<br/>Go to Sofa]
    
    SeqWork --> ActMove1
    SeqWork --> ActGrab
    SeqWork --> ActMove2
    
    %% 样式调整
    style SeqBat fill:#bbf,stroke:#333
    style SeqWork fill:#bbf,stroke:#333
    style CondBat fill:#ff9,stroke:#333,stroke-dasharray: 5 5
```    

In [8]:
import py_trees
import time
import random

In [9]:
# --- 0. 模拟机器人的黑板 (Blackboard) ---
# 用来存储机器人全局状态的数据中心
class RobotContext:
    def __init__(self):
        self.battery_level = 100
        self.location = "station"
        self.has_coffee = False

# 初始化全局上下文
robot = RobotContext()

In [10]:
# --- 1. 定义自定义行为节点 (Actions & Conditions) ---

class CheckBattery(py_trees.behaviour.Behaviour):
    """条件节点：检查电池是否低于阈值"""
    def __init__(self, name="Battery Low?"):
        super(CheckBattery, self).__init__(name)

    def update(self):
        # 模拟电量自然消耗
        robot.battery_level -= 5 
        print(f"   [系统检测] 当前电量: {robot.battery_level}%")
        
        if robot.battery_level < 20:
            print("   >>> 警告：电量过低！")
            return py_trees.common.Status.SUCCESS  # 条件满足（低电量）
        else:
            return py_trees.common.Status.FAILURE  # 条件不满足（电量充足）

class ChargeRobot(py_trees.behaviour.Behaviour):
    """动作节点：充电"""
    def __init__(self, name="Charging"):
        super(ChargeRobot, self).__init__(name)

    def update(self):
        if robot.battery_level < 100:
            robot.battery_level += 20
            print("   [动作] 正在充电中... (+20%)")
            return py_trees.common.Status.RUNNING  # 动作正在进行中
        else:
            print("   [动作] 充电完成！")
            return py_trees.common.Status.SUCCESS

class MoveTo(py_trees.behaviour.Behaviour):
    """动作节点：移动到某地"""
    def __init__(self, name, target_location):
        super(MoveTo, self).__init__(name)
        self.target = target_location
        self.steps = 0

    def initialise(self):
        # 每次进入该动作时重置计数器
        self.steps = 0
        print(f"   [导航] 开始前往: {self.target}")

    def update(self):
        if robot.location == self.target:
            return py_trees.common.Status.SUCCESS
        
        # 模拟移动需要时间（需要多次 tick）
        self.steps += 1
        print(f"   [导航] 正在移动中... ({self.steps}/3)")
        
        if self.steps >= 3:
            robot.location = self.target
            print(f"   [导航] 到达目的地: {self.target}")
            return py_trees.common.Status.SUCCESS
        
        return py_trees.common.Status.RUNNING

class GrabObject(py_trees.behaviour.Behaviour):
    """动作节点：抓取物体"""
    def __init__(self, name):
        super(GrabObject, self).__init__(name)

    def update(self):
        if robot.location != "kitchen":
            print("   [错误] 不在厨房，无法抓取！")
            return py_trees.common.Status.FAILURE
        
        print("   [机械臂] 成功抓取咖啡！")
        robot.has_coffee = True
        return py_trees.common.Status.SUCCESS

In [11]:
# --- 2. 构建行为树 ---

def create_tree():
    # 根节点：选择器（Selector）。它会优先执行左边的分支，如果左边成功或运行中，就忽略右边。
    root = py_trees.composites.Selector(name="Robot Brain", memory=False)

    # 分支1：电池保护（Sequence）
    # 逻辑：如果电池低（Success） -> 执行充电。如果电池不低（Failure） -> Sequence失败 -> Selector跳到下一个分支。
    battery_sequence = py_trees.composites.Sequence(name="Battery Guard", memory=False)
    battery_sequence.add_children([
        CheckBattery(),
        ChargeRobot()
    ])

    # 分支2：送咖啡任务（Sequence）
    # 这里设置 memory=True，意味着如果"去厨房"成功了，下次tick会直接从"抓咖啡"开始，不用重新去厨房。
    work_sequence = py_trees.composites.Sequence(name="Deliver Coffee", memory=True)
    work_sequence.add_children([
        MoveTo(name="Go to Kitchen", target_location="kitchen"),
        GrabObject(name="Pick Coffee"),
        MoveTo(name="Go to Sofa", target_location="sofa")
    ])

    # 将两个分支挂载到根节点
    root.add_children([battery_sequence, work_sequence])
    
    return root

In [12]:
# --- 3. 运行主循环 (The Tick Loop) ---

def run_simulation():
    tree = create_tree()
    # 初始化树（连接节点等）
    tree.setup(timeout=15)
    
    print("====== 模拟开始：机器人电量 100% ======")
    
    # 我们为了演示，手动设置电量快没电的情况，方便观察抢占行为
    robot.battery_level = 30 
    
    # 模拟 10 个时间步 (Ticks)
    for i in range(1, 11):
        print(f"\n--- Tick {i} ---")
        try:
            # 这里的 tick_once 是行为树的核心，它遍历树并调用 update()
            tree.tick_once()
            
            # 打印当前树中哪个节点在运行（可选，调试用）
            # print(py_trees.display.unicode_tree(root=tree, show_status=True))
            
        except KeyboardInterrupt:
            break
        time.sleep(0.5)

In [13]:
run_simulation()

====== 模拟开始：机器人电量 100% ======

--- Tick 1 ---
   [系统检测] 当前电量: 25%
   [导航] 开始前往: kitchen
   [导航] 正在移动中... (1/3)

--- Tick 2 ---
   [系统检测] 当前电量: 20%
   [导航] 正在移动中... (2/3)

--- Tick 3 ---
   [系统检测] 当前电量: 15%
   >>> 警告：电量过低！
   [动作] 正在充电中... (+20%)

--- Tick 4 ---
   [系统检测] 当前电量: 30%
   [导航] 开始前往: kitchen
   [导航] 正在移动中... (1/3)

--- Tick 5 ---
   [系统检测] 当前电量: 25%
   [导航] 正在移动中... (2/3)

--- Tick 6 ---
   [系统检测] 当前电量: 20%
   [导航] 正在移动中... (3/3)
   [导航] 到达目的地: kitchen
   [机械臂] 成功抓取咖啡！
   [导航] 开始前往: sofa
   [导航] 正在移动中... (1/3)

--- Tick 7 ---
   [系统检测] 当前电量: 15%
   >>> 警告：电量过低！
   [动作] 正在充电中... (+20%)

--- Tick 8 ---
   [系统检测] 当前电量: 30%
   [导航] 开始前往: kitchen
   [机械臂] 成功抓取咖啡！
   [导航] 开始前往: sofa
   [导航] 正在移动中... (1/3)

--- Tick 9 ---
   [系统检测] 当前电量: 25%
   [导航] 正在移动中... (2/3)

--- Tick 10 ---
   [系统检测] 当前电量: 20%
   [导航] 正在移动中... (3/3)
   [导航] 到达目的地: sofa
